In [ ]:
import pandas as pd
import os
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Filtrador de dataframes a partir de filtros
def dataframe_filter(df_original: pd.DataFrame, league: str = None, season: str = None, team: str = None, player: str = None):
    df = df_original.copy()

    # Filtramos por liga
    if league is not None:
        if "league" in df.columns:
            df = df[df['league'] == league]     # Filtro
    
    # Filtramos por temporada
    if season is not None:
        if "season" in df.columns:
            df = df[df['season'] == season]

    # Filtramos por equipo
    if team is not None:
        if "team_slug" in df.columns:
            team_slug = team.lower().replace(" ", "-")      # Conversioin a slug
            df = df[df['team_slug'] == team_slug]

    # Filtramos por jugador
    if player is not None:
        if "player_slug" in df.columns:
            player_slug = player.lower().replace(" ", "-")      # Conversioin a slug
            df = df[df['player_slug'] == player_slug]

    return df

# A partir de una temporada, obtenemos métricas relevantes para estudiar y comprarar las distintas ligas
def season_league_comparison_metrics(player_stats_summary_df: pd.DataFrame, goalkeeper_pct_df: pd.DataFrame, defender_pct_df: pd.DataFrame, midfielder_pct_df: pd.DataFrame, forward_pct_df: pd.DataFrame, season: str) -> pd.DataFrame:
    
    # Filtrado del dataframe
    season_df = dataframe_filter(df_original=player_stats_summary_df, season=season)

    # Selección de columnas
    season_df = season_df[['league','Minutes','expectedGoals','goals','attack_value_raw','bigChanceCreated','totalShots','def_actions','duelWon',
                        'aerialWon','fouls','interceptionWon','progressiveBallCarriesCount','passValueNormalized']]

    # Columnas a las cuales vamos a aplicar x90
    cols_per90 = ['expectedGoals','goals','attack_value_raw','bigChanceCreated','totalShots',
                'def_actions','duelWon','aerialWon','fouls','interceptionWon','progressiveBallCarriesCount']

    # Filtramos jugadores com 0 minutos (poco usual)
    season2526 = season_df[season_df['Minutes'] > 0].copy()

    # Calcular factor per90
    season2526[[c for c in cols_per90]] = (season_df[cols_per90].div(season_df['Minutes'], axis=0).mul(90))       # Mantenemos el mismo nombre

    # Aplicamos los nombres a las columnas
    season_df.columns = ['League','Minutes','ExpectedGoalsPer90','GoalsPer90','AttackValuePer90','BigChancesPer90','ShotsPer90',
                        'DefensiveActionsPer90','DuelsWonPer90','AerialWonPer90','FoulsPer90','InterceptionsPer90','ProgresionsPer90',
                        'PassValueNormalized']

    # Sacamos la columna de minutos
    season_df = season_df.drop(['Minutes'], axis=1)

    # Obtenemos las medidas medias por liga
    season_df = (season_df.groupby('League', as_index=False).mean(numeric_only=True))

    # Obtenemos otras métricas que vamos a usar
    season_df['AttackValue'] = season_df['ExpectedGoalsPer90'] + season_df['BigChancesPer90'] + season_df['AttackValuePer90']
    season_df['DeffenseValue'] = season_df['DefensiveActionsPer90'] + season_df['DuelsWonPer90'] + season_df['AerialWonPer90'] + season_df['InterceptionsPer90']
    season_df['ProgressionValue'] = season_df['ProgresionsPer90'] + season_df['PassValueNormalized']

    # Obtenemos el valor medio de cada variable de los percentiles por liga - función para hacerlo
    def get_season_percentiles_medians(df: pd.DataFrame, season: str) -> pd.DataFrame:

        # Filtramos por liga
        df_filtered = dataframe_filter(df_original=df, season=season).drop(['season', 'player_name', 'player_slug'], axis=1)

        # Agrupamos por liga - obtenemos la mediana
        df_filtered = (df_filtered.groupby('league', as_index=False).median(numeric_only=True))

        return df_filtered

    # Calculamos para cada posición las medianas y cambiamos los nombres en las colunnas
    mean_league_gk_df = get_season_percentiles_medians(df=goalkeeper_pct_df, season=season)
    mean_league_gk_df.columns = ['League','PctShotStoppingGK','PctReliabilityGK','PctAreaControlGK','PctSweeperKeeperGK','PctBuildUpPlayGK']
    mean_league_df_df = get_season_percentiles_medians(df=defender_pct_df, season=season)
    mean_league_df_df.columns = ['League','PctDefensiveActionsDF','PctDuelsDF','PctAerialAbilityDF','PctBuildUpPlayDF','PctDefensiveReliabilityDF']
    mean_league_md_df = get_season_percentiles_medians(df=midfielder_pct_df, season=season)
    mean_league_md_df.columns = ['League','PctBallDistributionMD','PctProgressionMD','PctChanceCreationMD','PctDefensiveBalanceMD','PctBallRetentionMD']
    mean_league_fw_df = get_season_percentiles_medians(df=forward_pct_df, season=season)
    mean_league_fw_df.columns = ['League','PctFinishingFW','PctChanceCreationFW','PctThreatFW','PctOffBallInvolvementFW','PctEfficiencyFW']

    # Unimos el dataframe de información de la liga con la de sus posiciones en uno general
    all_league_info = pd.merge(season_df, mean_league_gk_df, on='League')
    all_league_info = pd.merge(all_league_info, mean_league_df_df, on='League')
    all_league_info = pd.merge(all_league_info, mean_league_md_df, on='League')
    all_league_info = pd.merge(all_league_info, mean_league_fw_df, on='League')

    # Colores de las ligas para diferenciar
    LEAGUE_COLORS = {"Bundesliga": "#D20515", "La Liga": "#FF8C00", "Ligue 1": "#0055A4",
                     "Premier League": "#6A0DAD", "Serie A": "#008C45"}
    all_league_info['Color'] = all_league_info['League'].map(LEAGUE_COLORS)

    return all_league_info

# Creación de un scatterplot de amenaza ofensiva de la liga
def offensive_thread_scatter_creation(df: pd.DataFrame, season: str, figures_path: str = None):

    # SCATTERPLOT PARA LA AMENAZA OFENSIVA PARA CADA LIGA - Compara goles esperados con tiros y goles finales
    scatter_offensive_thread = px.scatter(df, x="ExpectedGoalsPer90", y="ShotsPer90", color="GoalsPer90",     # Color gradiente
                                          text="League", color_continuous_scale="Reds",  size_max=60)

    # Mostramos un texto con la liga encima del punto para una mejor comprensión
    scatter_offensive_thread.update_traces(textposition="top center", marker=dict(size=20, line=dict(width=1)))

    # Layout de título y ejes
    scatter_offensive_thread.update_layout(
        title=dict(text=f"AMENAZA OFENSIVA DE LAS CINCO GRANDES LIGAS - TEMPORADA {season}<br><sup> Tiros por 90 vs. Goles esperados por 90 vs. Goles por 90 (media por jugador)</sup>",  # Subtitulo
                   x=0.5),
        xaxis_title="Goles esperados por 90", yaxis_title="Tiros por 90",
        coloraxis_colorbar=dict(title="Goles por 90"))
    
    # Guardamos si el path no es nulo
    if figures_path is not None:
        name_figure = f'LeaguesAttackingThreadComparison{season.replace('/','')}.png'       # Nombre de la figura   
        final_figure_path = os.path.join(figures_path, name_figure)                         # Nombre final del path a guardar la figura
        scatter_offensive_thread.write_image(final_figure_path, width=1400, height=800, scale=2)    # Guardado de la figura

    return scatter_offensive_thread

# Comparación de las ligas más físicas a partir de valores de duelos
def physicality_map_creation(df: pd.DataFrame, season: str, figures_path: str = None): 

    # SCATTERPLOT DE POTENCIA FÍSICA POR LIGA - compara duelos y duelos aereos ganados
    scatter_physicality = px.scatter(df, x="DuelsWonPer90", y="AerialWonPer90", color="Color",     # Color de la lga
                                     text="League", size_max=60)

    # Mostramos un texto con la liga encima del punto para una mejor comprensión
    scatter_physicality.update_traces(textposition="top center", marker=dict(size=20, line=dict(width=1)))

    # Layout de título y ejes
    scatter_physicality.update_layout(
        title=dict(text=f"MAPA DE PORTENTO FÍSICO POR LIGA - TEMPORADA {season}<br><sup> Duelos ganados por 90 vs. Duelos aereos (media por jugador)</sup>",  # Subtitulo
                   x=0.5),
        xaxis_title="Duelos ganados por 90", yaxis_title="Duelos aereos por 90", showlegend=False)
    
    # Guardamos si el path no es nulo
    if figures_path is not None:
        name_figure = f'LeaguesPhysicalityMap{season.replace('/','')}.png'       # Nombre de la figura   
        final_figure_path = os.path.join(figures_path, name_figure)                         # Nombre final del path a guardar la figura
        scatter_physicality.write_image(final_figure_path, width=1400, height=800, scale=2)    # Guardado de la figura

    return scatter_physicality

# Comparación del estilo físico de las ligas
def tactical_style_creation(df: pd.DataFrame, season: str, figures_path: str = None): 

    # SCATTERPLOT DE ESTILO TÁCTICO POR LIGA - comparación de valores defensivos y ofensivos
    scatter_tact_style = px.scatter(df, x="DeffenseValue", y="AttackValue", color="Color",     # Color de la lga
                                    text="League", size_max=60)

    # Mostramos un texto con la liga encima del punto para una mejor comprensión
    scatter_tact_style.update_traces(textposition="top center", marker=dict(size=20, line=dict(width=1)))

    # Layout de título y ejes
    scatter_tact_style.update_layout(
        title=dict(text=f"ESTILO TÁCTICO POR LIGA - TEMPORADA {season}<br><sup> Comparación de métricas defensivas y ofensivas a partir de los datos de los jugadores</sup>",  # Subtitulo
                   x=0.5),
        xaxis_title="Valor defensivo", yaxis_title="Valor ofensivo", showlegend=False)
    
    # Guardamos si el path no es nulo
    if figures_path is not None:
        name_figure = f'LeaguesTacticalStyle{season.replace('/','')}.png'       # Nombre de la figura   
        final_figure_path = os.path.join(figures_path, name_figure)                         # Nombre final del path a guardar la figura
        scatter_tact_style.write_image(final_figure_path, width=1400, height=800, scale=2)    # Guardado de la figura

    return scatter_tact_style

# Creación de un gráfico de radar para mostrar los percentiles por cada posición y liga
def radar_5metrics_creation(df: pd.DataFrame, position: str, season: str, title: str, figures_path: str = None):

    # Labels según posición
    if position == 'Goalkeeper':
        labels = {'PctShotStoppingGK': 'Shot Stopping',
                'PctReliabilityGK': 'Reliability', 
                'PctAreaControlGK': 'Area Control', 
                'PctSweeperKeeperGK': 'Sweeper Keeper',
                'PctBuildUpPlayGK': 'Build-up Play'}
        
    elif position == 'Defender':
        labels = {'PctDefensiveActionsDF': 'Defensive Actions',
                'PctDuelsDF': 'Duels', 
                'PctAerialAbilityDF': 'Aerial Ability', 
                'PctBuildUpPlayDF': 'Build-up Play',
                'PctDefensiveReliabilityDF': 'Defensive Reliability'}
        
    elif position == 'Midfielder':
        labels = {'PctBallDistributionMD': 'Ball Distribution',
                'PctProgressionMD': 'Progression', 
                'PctChanceCreationMD': 'Chance Creation', 
                'PctDefensiveBalanceMD': 'Defensive Balance',
                'PctBallRetentionMD': 'Ball Retention'}

    elif position == 'Forward':
        labels = {'PctFinishingFW': 'Finishing',
                'PctChanceCreationFW': 'Chance Creation',
                'PctThreatFW': 'Threat', 
                'PctOffBallInvolvementFW': 'Ball Involvement', 
                'PctEfficiencyFW': 'Efficiency'}

    # Metricas
    metrics = list(labels.keys())

    # Categorias a mostrar
    categories = [labels.get(m, m) for m in metrics]
    categories = categories + [categories[0]]

    # Calculamos el rango dinámico del radar para mejorar la comprensión
    global_min = df[metrics].min().min() - 0.01
    global_max = df[metrics].max().max() + 0.01

    # Creación de la figura
    radar = go.Figure()

    # Una línea para cada liga
    for _, row in df.iterrows():
        values = [row[m] for m in metrics] + [row[metrics[0]]]
        trace_kwargs = dict(r=values, theta=categories, name=str(row['League']), mode="lines+markers", line=dict(width=2), marker=dict(size=6))

        # Pintamos de color
        trace_kwargs["line"]["color"] = row['Color']
        trace_kwargs["marker"]["color"] = row['Color']

        radar.add_trace(go.Scatterpolar(**trace_kwargs))

    # Títulos del gráfico
    radar.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        polar=dict(radialaxis=dict(visible=True,range=[global_min, global_max])),
        showlegend=True,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        margin=dict(b=100))
    
    # Guardamos si el path no es nulo
    if figures_path is not None:
        name_figure = f'Leagues{position}RadarChart{season.replace('/','')}.png'       # Nombre de la figura   
        final_figure_path = os.path.join(figures_path, name_figure)                         # Nombre final del path a guardar la figura
        radar.write_image(final_figure_path, width=1400, height=800, scale=2)    # Guardado de la figura
    
    return radar

# Creación de un violin plot de la metrica que queramos
def violin_plot_creation(df: pd.DataFrame, metric: str, metric_title: str, season: str, title: str, subtitle: str, figures_path: str = None):

    # Ligas, colores
    leagues = ['Serie A', 'Bundesliga', 'La Liga', 'Ligue 1', 'Premier League']
    border_color = {"Bundesliga": "rgba(210, 5, 21, 1)", "La Liga": "rgba(255, 140, 0, 1)", "Ligue 1": "rgba(0, 85, 164, 1)",
                    "Premier League": "rgba(106, 13, 173, 1)", "Serie A": "rgba(0, 140, 69, 1)"}
    fill_color = {"Bundesliga": "rgba(210, 5, 21, 0.3)", "La Liga": "rgba(255, 140, 0, 0.3)", "Ligue 1": "rgba(0, 85, 164, 0.3)",
                  "Premier League": "rgba(106, 13, 173, 0.3)", "Serie A": "rgba(0, 140, 69, 0.3)"}
    
    # Creación del violin plot
    violin = go.Figure()

    # Por liga creamos el violin
    for league in leagues:

        # Info
        df_league = df[df['league'] == league]
        violin.add_trace(go.Violin(x=df_league['league'], y=df_league[metric], name=league, box_visible=True, 
                         meanline_visible=True, fillcolor=fill_color[league], line_color=border_color[league]))
        
    # Layout
    violin.update_layout(title=dict(text=f"{title}<br><sup> {subtitle}</sup>", x=0.5, xanchor="center"), yaxis_title='Value')

    # Guardamos si el path no es nulo
    if figures_path is not None:
        name_figure = f'Leagues{metric_title}ViolinChart{season.replace('/','')}.png'       # Nombre de la figura   
        final_figure_path = os.path.join(figures_path, name_figure)                         # Nombre final del path a guardar la figura
        violin.write_image(final_figure_path, width=1400, height=800, scale=2)    # Guardado de la figura
    
    return violin

In [ ]:
data_path = "G:\\FootballData\\data"
proc_data_path = os.path.join(data_path, 'clean')
figures_path = os.path.join(data_path, 'images')

# Lectura de todos los datos que necesitaremos
match_info_df = pd.read_csv(os.path.join(proc_data_path, "MatchInfo.csv"), sep=';')
player_info_df = pd.read_csv(os.path.join(proc_data_path, "PlayerInfo.csv"), sep=';')
player_stats_df = pd.read_csv(os.path.join(proc_data_path, "PlayerStats.csv"), sep=';')
player_stats_summary_df = pd.read_csv(os.path.join(proc_data_path, "PlayerStatsSummary.csv"), sep=';')
player_team_stats_summary_df = pd.read_csv(os.path.join(proc_data_path, "PlayerTeamStatsSummary.csv"), sep=';')
defender_pct_df = pd.read_csv(os.path.join(proc_data_path, "DefenderPercentile.csv"), sep=';')
midfielder_pct_df = pd.read_csv(os.path.join(proc_data_path, "MidfielderPercentile.csv"), sep=';')
forward_pct_df = pd.read_csv(os.path.join(proc_data_path, "ForwardPercentile.csv"), sep=';')
goalkeeper_pct_df = pd.read_csv(os.path.join(proc_data_path, "GoalkeeperPercentile.csv"), sep=';')

In [ ]:
season = '24/25'

# Obtenemos el dataframe que nos va a ayudar a crear las visualizaciones
df = season_league_comparison_metrics(season=season)

# Creación de los gráficos scatter simples
off_thread_sctt = offensive_thread_scatter_creation(df=df, season=season, figures_path=figures_path)
phy_map_sctt = physicality_map_creation(df=df, season=season, figures_path=figures_path)
tact_style_sctt = tactical_style_creation(df=df, season=season, figures_path=figures_path)

# Para cada posición, creamos los radar charts
gk_radar_chart = radar_5metrics_creation(df=df, position='Goalkeeper', season=season, figures_path=figures_path,
                                         title='Comparación de los percentiles medios por jugador en distintas facetas del juego (Porteros)')
df_radar_chart = radar_5metrics_creation(df=df, position='Defender', season=season, figures_path=figures_path,
                                         title='Comparación de los percentiles medios por jugador en distintas facetas del juego (Defensas)')
mf_radar_chart = radar_5metrics_creation(df=df, position='Midfielder', season=season, figures_path=figures_path,
                                         title='Comparación de los percentiles medios por jugador en distintas facetas del juego (Centrocampistas)')
fw_radar_chart = radar_5metrics_creation(df=df, position='Forward', season=season, figures_path=figures_path,
                                         title='Comparación de los percentiles medios por jugador en distintas facetas del juego (Delanteros)')

# Para cada métrica de cada una de las posiciones, obtenemos las gráficas de violines para ver la liga con los mejores jugadores en cada ambito